# Spectral Pre-Analysis for MSI Apple Bruise Detection

This notebook performs spectral correlation analysis on 9-band multispectral imaging (MSI) data
to understand the spectral differences between normal and bruised apple tissue.

**Outputs:**
- 9×9 Pearson correlation matrix for normal regions
- 9×9 Pearson correlation matrix for bruise regions
- Difference matrix ΔR = R_normal - R_bruise
- Spectral mean curves (normal vs. bruise)

All results are saved to `outputs/spectral_analysis/`.

In [ ]:
import os
import sys
import json

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yaml

# Add project root to path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, PROJECT_ROOT)

from utils.spectral_analysis import run_spectral_analysis, collect_spectral_pixels, compute_correlation_matrix

# Load config
config_path = os.path.join(PROJECT_ROOT, "configs", "config.yaml")
with open(config_path, "r") as f:
    cfg = yaml.safe_load(f)

print("Config loaded.")
print(f"  Image dir:  {cfg['data']['image_dir']}")
print(f"  Mask dir:   {cfg['data']['mask_dir']}")
print(f"  Wavelengths: {cfg['data']['wavelengths']}")

In [ ]:
# ========== 📂 DATA INPUT PATHS ==========
image_dir = os.path.join(PROJECT_ROOT, cfg["data"]["image_dir"])
mask_dir = os.path.join(PROJECT_ROOT, cfg["data"]["mask_dir"])
output_dir = os.path.join(PROJECT_ROOT, "outputs", "spectral_analysis")
wavelengths = cfg["data"]["wavelengths"]
# ==========================================

# Scan available samples
sample_names = sorted([f[:-4] for f in os.listdir(image_dir) if f.endswith(".npy")])
print(f"Found {len(sample_names)} samples")

# Run full spectral analysis
results = run_spectral_analysis(
    image_dir=image_dir,
    mask_dir=mask_dir,
    sample_names=sample_names,
    output_dir=output_dir,
    wavelengths=wavelengths,
)

print(f"\nNormal pixels collected: {results['num_normal_pixels']}")
print(f"Bruise pixels collected: {results['num_bruise_pixels']}")

In [ ]:
# Display correlation matrices side by side
wl_labels = [str(w) for w in wavelengths]
R_normal = np.array(results["R_normal"])
R_bruise = np.array(results["R_bruise"])
delta_R = np.array(results["delta_R"])

fig, axes = plt.subplots(1, 3, figsize=(22, 6))

sns.heatmap(R_normal, annot=True, fmt=".2f", xticklabels=wl_labels,
            yticklabels=wl_labels, cmap="coolwarm", vmin=-1, vmax=1, ax=axes[0])
axes[0].set_title("Normal Region Correlation")

sns.heatmap(R_bruise, annot=True, fmt=".2f", xticklabels=wl_labels,
            yticklabels=wl_labels, cmap="coolwarm", vmin=-1, vmax=1, ax=axes[1])
axes[1].set_title("Bruise Region Correlation")

sns.heatmap(delta_R, annot=True, fmt=".2f", xticklabels=wl_labels,
            yticklabels=wl_labels, cmap="RdBu_r", center=0, ax=axes[2])
axes[2].set_title("ΔR = R_normal - R_bruise")

fig.tight_layout()
plt.show()

In [ ]:
# Display spectral mean curves
normal_mean = np.array(results["normal_mean"])
normal_std = np.array(results["normal_std"])
bruise_mean = np.array(results["bruise_mean"])
bruise_std = np.array(results["bruise_std"])

x = np.arange(len(wavelengths))

fig, ax = plt.subplots(figsize=(10, 6))
ax.errorbar(x, normal_mean, yerr=normal_std, label="Normal", marker="o", capsize=4, linewidth=2)
ax.errorbar(x, bruise_mean, yerr=bruise_std, label="Bruise", marker="s", capsize=4, linewidth=2)
ax.set_xticks(x)
ax.set_xticklabels(wl_labels, rotation=45)
ax.set_xlabel("Wavelength (nm)", fontsize=12)
ax.set_ylabel("Mean Reflectance", fontsize=12)
ax.set_title("Spectral Mean Curves: Normal vs. Bruise", fontsize=14)
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

# Print spectral difference
print("\nSpectral difference (Normal - Bruise):")
for i, wl in enumerate(wavelengths):
    diff = normal_mean[i] - bruise_mean[i]
    print(f"  {wl} nm: Δ = {diff:+.6f}")

## Interpretation

**What to look for:**

1. **Correlation matrices:** If bruise regions show lower inter-band correlation compared to normal regions, this supports the hypothesis that bruising disrupts the spectral structure — which is exactly what LSAA (Local Spectral Anomaly Attention) is designed to exploit.

2. **ΔR matrix:** Large absolute values in the difference matrix indicate wavelength pairs where the correlation structure changes most between normal and bruised tissue. These are the spectral "signatures" of bruising.

3. **Spectral curves:** Bands where the normal-vs-bruise separation is largest are the most discriminative for bruise detection. The LSAA module should learn to focus on these bands.

**Files saved:**
- `outputs/spectral_analysis/corr_normal.png`
- `outputs/spectral_analysis/corr_bruise.png`
- `outputs/spectral_analysis/corr_delta.png`
- `outputs/spectral_analysis/spectral_curves.png`
- `outputs/spectral_analysis/spectral_analysis.json`